# Data Preparation: MD Trajectory
Instead of static structures from the Materials Project (which do not have forces), 
we generate a small Molecular Dynamics (MD) trajectory of a Copper supercell using ASE's built-in EMT calculator. 
This ensures our dataset contains highly correlated structures with **real, non-zero forces**, which is essential for benchmarking force-matching potentials like ACE and MACE.

In [1]:
import os
import numpy as np
from ase.build import bulk
from ase.calculators.emt import EMT
from ase.md.velocitydistribution import MaxwellBoltzmannDistribution
from ase.md.verlet import VelocityVerlet
from ase.io import write
from ase.calculators.singlepoint import SinglePointCalculator
from ase import units
from tqdm.auto import tqdm

os.makedirs('../data', exist_ok=True)
np.random.seed(42)

# 1. Setup Initial Structure and Calculator
print("Setting up Cu supercell...")
atoms = bulk('Cu', 'fcc', a=3.6) * (2, 2, 2)
atoms.calc = EMT()

# 2. Initialize MD (Temperature = 300K)
MaxwellBoltzmannDistribution(atoms, temperature_K=300)
dyn = VelocityVerlet(atoms, 1.0 * units.fs)

# 3. Run MD and collect 1400 samples
n_samples = 1400
dataset = []

print(f"Running MD simulation to generate {n_samples} frames...")
for _ in tqdm(range(n_samples), desc="MD Steps"):
    # Run a few steps between sampling to decorrelate slightly
    dyn.run(10)
    
    # Extract current state
    new_atoms = atoms.copy()
    energy = atoms.get_potential_energy()
    forces = atoms.get_forces()
    
    # Store energy and forces permanently via SinglePointCalculator
    calc = SinglePointCalculator(new_atoms, energy=energy, forces=forces)
    new_atoms.calc = calc
    
    dataset.append(new_atoms)

Setting up Cu supercell...
Running MD simulation to generate 1400 frames...


MD Steps:   0%|          | 0/1400 [00:00<?, ?it/s]

In [2]:
import random
random.seed(42)

# Shuffle the trajectory so train/val/test are drawn from the entire distribution
random.shuffle(dataset)

train_data = dataset[:1000]
val_data   = dataset[1000:1200]
test_data  = dataset[1200:]

write('../data/train.extxyz', train_data)
write('../data/val.extxyz',   val_data)
write('../data/test.extxyz',  test_data)

print('Data Splitting Complete!')
print(f'Train: {len(train_data)} | Val: {len(val_data)} | Test: {len(test_data)}')

Data Splitting Complete!
Train: 1000 | Val: 200 | Test: 200
